# EchoClause — Build with Gemma Hackathon Demo

Evidence-grounded promise-to-contract reconciliation using **Gemma 4** multimodal extraction and deterministic calculators.

> EchoClause compares representations across supplied evidence. It does not provide legal advice or determine legal enforceability.

In [ ]:
# Cell 1 — Configuration
RUN_FULL_BENCHMARK = False  # Keep False for hackathon demo
PROJECT_DIR = "/kaggle/working"
MODEL_PRIMARY = "google/gemma-4-E4B-it"
MODEL_FALLBACK = "google/gemma-4-E2B-it"

In [ ]:
# Cell 2 — Install dependencies (kernel bundle includes full project at /kaggle/working)
import os, subprocess, sys
from pathlib import Path

DEST = Path(PROJECT_DIR)
os.chdir(DEST)
for k in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY"):
    os.environ.pop(k, None)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev,gemma]"])
print("Installed at", DEST)

In [ ]:
# Cell 3 — Generate demo assets
import subprocess, sys
subprocess.check_call([sys.executable, "scripts/generate_demo_assets.py"])

In [ ]:
# Cell 4 — R1 Runtime spike (Gemma multimodal + function calling)
import subprocess, sys
result = subprocess.run([sys.executable, "scripts/run_runtime_spike.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
assert result.returncode == 0, "R1 runtime spike failed"

In [ ]:
# Cell 5 — Full pipeline (live Gemma extraction)
import sys
sys.path.insert(0, str(DEST))
from echo_clause.gemma_runtime import GemmaRuntime
from echo_clause.pipeline import run_pipeline, write_pipeline_artifact

runtime = GemmaRuntime()
loaded = runtime.load()
if not loaded:
    print("Live load failed — using recorded replay for pipeline validation")
    report = run_pipeline(use_recorded=True)
else:
    report = run_pipeline(runtime=runtime, audio_fallback=True)
artifact = write_pipeline_artifact(report, prefix="kaggle_pipeline")
print(f"Model: {report.get('model_id')}")
print(f"Conflicts: {report['conflict_count']}")
print(f"Gold: {report['demo_validation']}")
print(f"Artifact: {artifact}")

In [ ]:
# Cell 6 — Validate against gold.json (must detect 5/5 contradictions)
validation = report["demo_validation"]
assert validation["all_gold_detected"], f"Missing: {validation.get('missing_fields')}"
assert validation["detected_gold_contradictions"] >= 5
print("PASS: 5/5 gold contradictions detected")

In [ ]:
# Cell 7 — Unit tests (skip extended benchmark unless enabled)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pytest", "-q", "--ignore=tests/test_benchmark.py" if not RUN_FULL_BENCHMARK else "-q"])
print("All tests passed")

## Results

EchoClause extracted claims from advertisement, sales audio, support chat, and contract images, normalized financial terms deterministically, and flagged contradictions before signing.